### Imports and setup

In [2]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, precision_score, recall_score
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import Trainer, TrainingArguments

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cpu


### Load train/validation/test splits

In [3]:
project_root = Path("..").resolve()
split_dir = project_root / "data" / "processed" / "splits"
processed_dir = project_root / "data" / "processed"

train_df = pd.read_csv(split_dir / "train.csv")
val_df = pd.read_csv(split_dir / "val.csv")
test_df = pd.read_csv(split_dir / "test.csv")

target_cols = sorted([c for c in train_df.columns if c.startswith("y_")])
if not target_cols:
    raise ValueError("No target columns found (expected columns starting with 'y_').")

for df in (train_df, val_df, test_df):
    if "text_clean" not in df.columns:
        raise ValueError("Missing required column: text_clean")
    df["text_clean"] = df["text_clean"].fillna("").astype(str)

print({"train": len(train_df), "val": len(val_df), "test": len(test_df)})
print("Labels:", len(target_cols))

{'train': 129, 'val': 27, 'test': 29}
Labels: 6


### Tokenize text and build datasets

In [17]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
MAX_LEN = 256

def encode_text(series):
    return tokenizer(
        series.tolist(),
        truncation=True,
        padding=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )

train_enc = encode_text(train_df["text_clean"])
val_enc = encode_text(val_df["text_clean"])
test_enc = encode_text(test_df["text_clean"])

train_labels = torch.tensor(train_df[target_cols].values, dtype=torch.float32)
val_labels = torch.tensor(val_df[target_cols].values, dtype=torch.float32)
test_labels = torch.tensor(test_df[target_cols].values, dtype=torch.float32)

class EmailDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = EmailDataset(train_enc, train_labels)
val_ds = EmailDataset(val_enc, val_labels)
test_ds = EmailDataset(test_enc, test_labels)

### Configure model and training process

In [21]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(target_cols),
    problem_type="multi_label_classification",
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "precision_micro": precision_score(labels, preds, average="micro", zero_division=0),
        "recall_micro": recall_score(labels, preds, average="micro", zero_division=0),
    }

# Per-label positive class weights for BCEWithLogitsLoss.
label_counts = train_df[target_cols].sum(axis=0).astype(float).values
num_samples = float(len(train_df))
pos_weight = (num_samples - label_counts) / np.clip(label_counts, 1.0, None)
pos_weight = torch.tensor(pos_weight, dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.pos_weight = pos_weight

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fct(logits, labels.float())
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=str(project_root / "models" / "distilbert_multilabel_run"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    seed=SEED,
    report_to="none",
    disable_tqdm=True,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    pos_weight=pos_weight,
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 747.74it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Train, evaluate, and save artifacts

In [22]:
train_output = trainer.train()
print(train_output)

val_metrics = trainer.evaluate(eval_dataset=val_ds)
test_metrics = trainer.evaluate(eval_dataset=test_ds)
print("Validation:", val_metrics)
print("Test:", test_metrics)

# Simple threshold tuning
val_pred = trainer.predict(val_ds)
val_probs = 1 / (1 + np.exp(-val_pred.predictions))
val_true = val_pred.label_ids

# Try thresholds from 0.15 to 0.60 in steps of 0.05, and pick the one with best micro-F1 on validation set.
threshold_grid = np.arange(0.15, 0.61, 0.05)
grid_scores = [
    f1_score(val_true, (val_probs >= t).astype(int), average="micro", zero_division=0)
    for t in threshold_grid
]
best_threshold = float(threshold_grid[int(np.argmax(grid_scores))])
print("Best global threshold (validation):", best_threshold)

val_preds_tuned = (val_probs >= best_threshold).astype(int)
print("Validation positive prediction rate:", float(val_preds_tuned.mean()))

test_pred = trainer.predict(test_ds)
test_probs = 1 / (1 + np.exp(-test_pred.predictions))
test_true = test_pred.label_ids
test_preds_tuned = (test_probs >= best_threshold).astype(int)
print("Test positive prediction rate:", float(test_preds_tuned.mean()))

print(
    "Test tuned metrics:",
    {
        "f1_micro": f1_score(test_true, test_preds_tuned, average="micro", zero_division=0),
        "f1_macro": f1_score(test_true, test_preds_tuned, average="macro", zero_division=0),
        "precision_micro": precision_score(test_true, test_preds_tuned, average="micro", zero_division=0),
        "recall_micro": recall_score(test_true, test_preds_tuned, average="micro", zero_division=0),
    },
)

final_model_dir = project_root / "models" / "distilbert_multilabel"
final_model_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(final_model_dir))
tokenizer.save_pretrained(str(final_model_dir))

pd.DataFrame({
    "label": [c.replace("y_", "") for c in target_cols],
    "label_id": list(range(len(target_cols))),
}).to_csv(processed_dir / "label_taxonomy.csv", index=False)

pd.DataFrame({
    "label": [c.replace("y_", "") for c in target_cols],
    "label_id": list(range(len(target_cols))),
    "threshold": [best_threshold] * len(target_cols),
}).to_csv(processed_dir / "label_thresholds.csv", index=False)

print("Saved model to:", final_model_dir)
print("Saved thresholds to:", processed_dir / "label_thresholds.csv")

c:\Users\willd\Dissertation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '1.074', 'eval_f1_micro': '0.4132', 'eval_f1_macro': '0.3924', 'eval_precision_micro': '0.2778', 'eval_recall_micro': '0.8065', 'eval_runtime': '6.084', 'eval_samples_per_second': '4.438', 'eval_steps_per_second': '0.658', 'epoch': '1'}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]
c:\Users\willd\Dissertation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


{'eval_loss': '1.05', 'eval_f1_micro': '0.4118', 'eval_f1_macro': '0.4367', 'eval_precision_micro': '0.2958', 'eval_recall_micro': '0.6774', 'eval_runtime': '8.608', 'eval_samples_per_second': '3.137', 'eval_steps_per_second': '0.465', 'epoch': '2'}


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.16s/it]
c:\Users\willd\Dissertation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


...

Best global threshold (validation): 0.5500000000000002
Validation positive prediction rate: 0.15432098765432098


c:\Users\willd\Dissertation\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Test positive prediction rate: 0.13793103448275862
Test tuned metrics: {'f1_micro': 0.5283018867924528, 'f1_macro': 0.5371591371591371, 'precision_micro': 0.5833333333333334, 'recall_micro': 0.4827586206896552}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.52it/s]


Saved model to: C:\Users\willd\Dissertation\models\distilbert_multilabel
Saved thresholds to: C:\Users\willd\Dissertation\data\processed\label_thresholds.csv
